In [40]:
from dotenv import load_dotenv
from langgraph.graph import START,END,StateGraph,add_messages
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import ToolNode,tools_condition
from langchain.tools import tool
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.checkpoint.memory import InMemorySaver


load_dotenv()

True

In [41]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
checkpointer=InMemorySaver()


In [42]:
class MessageState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

@tool
def calculator(a:int,b:int,oprn:str):
    """
        This is the calculator tool that takes 3 arguments
        a and b as a number and oprn which supports:
        add,minus,multi,div
        and each of them returns the result
    """
    if(oprn =="add"):
        return a+b 
    elif (oprn == "minus"):
        return a-b 
    elif (oprn =="multi"):
        return a*b 

    elif (oprn =="div"):
        return a/b 
    else :
        return None
    

search_tools = DuckDuckGoSearchRun()

tools = ToolNode([calculator,search_tools])
llm_with_tools = llm.bind_tools([calculator,search_tools])




In [ ]:

def chatNode(state:MessageState):

    query = state["messages"]

    response = llm_with_tools.invoke(query)

    return {"messages":[response]}


graph = StateGraph(MessageState)

graph.add_node("chat",chatNode)
graph.add_node("tools",tools)

graph.add_edge(START,"chat")
graph.add_conditional_edges("chat",tools_condition)
graph.add_edge("tools","chat")



In [50]:
worflow = graph.compile(checkpointer=checkpointer)

In [65]:


while True:
    q = input("enter message : ")
    print("USER: ",q)
    results = worflow.invoke({"messages":HumanMessage(content=q)},config={"configurable":{"thread_id":2}})
    res = results["messages"][-1].content
    print("AI: ",res)





USER:  hi
AI:  Hello! How can I help you today?
USER:  who is winner of ipl 2026 
AI:  [{'type': 'text', 'text': "I'm sorry, but I cannot tell you who will win the IPL 2026 as it is a future event.", 'extras': {'signature': 'CoQCAWkUfROxIROnlAA+MCVBKHVswLCbbn3MybkK9RT7UhpGrCZqkwakvuzAAif3wQgos6e4uEYmwyL3lvUef1iELIXLbWPUWHPhniqDUfhrVRghJ4g4VG3NohFajcmMjbX08y6a8y7I3MyqhkGjPXsQDWBE5N4Y2XYHduSjXAp0bEYVFhCeYxsnToKXociHyY246twtq+PklLtnb5tetvYKhIEcthm4Xc5E2rvGU96vIeIEtSJMkGw2NCc0VqS0cgdeRPgBKCyOtKtiFYj5qbqz5mIiwr+VQzMq8gjOnZi8BSi3JjrJhCuDScpCykdzQr8r6qULyf+GsQpXu3MtoP2ETfKFP8A='}}]
USER:  30303030 muliply 7818181
AI:  The product of 30303030 and 7818181 is 236914573388430.
USER:  full form of cpu 


KeyboardInterrupt: 

The winner of IPL 2026 is not available yet, as the event is in the future. However, Royal Challengers Bengaluru has won the IPL title in 2025 by dominating Gujarat Titans in the final. They also won in 2024, making it back-to-back titles.
